In [1]:
import pandas as pd
import numpy as np
import glob
import os

print("All imports successful!")

All imports successful!


In [2]:
q1 = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_2004Q1.txt", sep = "|", header = None)
q2 = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_2004Q2.txt", sep = "|", header = None)
q3 = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_2004Q3.txt", sep = "|", header = None)
q4 = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_2004Q4.txt", sep = "|", header = None)

In [3]:
orig_2004 = pd.concat([q1, q2, q3, q4], ignore_index= True)
orig_2004.shape

(1677203, 32)

In [4]:
column_names = ["cred_score", "first_payment_date", "first_time_homebuyer_flag", "maturity_date", 
                "metro_code_msa", "mortgage_insurance_percent", "unit_no", "occupancy_status",
                "og_cltv", "og_dti", "og_upb", "og_ltv",
                "og_int_rate", "channel", "ppm_flag", "amort_type",
                "prop_state", "prop_type", "postal_code", "loan_seq_no",
                "loan_purpose", "og_loan_term", "no_of_borrowers", "seller_name",
                "servicer_name", "super_conforming_flag", "pre_relief_refinance_loan_seq_no", "special_elig_program",
                "relief_refinance_indicator", "prop_val_method", "int_only_indicator", "MI_cal_indicator"
                ]

In [5]:
orig_2004.columns = column_names

In [6]:
orig_2004.head()

,cred_score,first_payment_date,first_time_homebuyer_flag,maturity_date,metro_code_msa,mortgage_insurance_percent,unit_no,occupancy_status,og_cltv,og_dti,...,no_of_borrowers,seller_name,servicer_name,super_conforming_flag,pre_relief_refinance_loan_seq_no,special_elig_program,relief_refinance_indicator,prop_val_method,int_only_indicator,MI_cal_indicator
0,755,200403,N,203402,NaN,0,1,P,80,24,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9
1,623,200403,N,201902,NaN,0,1,P,74,18,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9
2,724,200403,Y,203402,45060.0,30,1,P,95,39,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9
3,767,200404,N,203403,29420.0,0,1,S,80,24,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9
4,680,200403,Y,201902,NaN,25,1,P,93,28,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,9


### 1. Check for Null values.

In [7]:
na_val = orig_2004.isnull().sum()
print(na_val[na_val>0]/len(orig_2004))

metro_code_msa                      0.43724
super_conforming_flag               1.00000
pre_relief_refinance_loan_seq_no    1.00000
relief_refinance_indicator          1.00000
dtype: float64


In [8]:
# 4 columns with missing values.
# super_conforming_flag, pre_relief_refinance_loan_seq_no and
# relief_refinance_indicator has 100 percent missing values so it will best tp drop these 3 columns.

orig_2004.drop( columns = ["super_conforming_flag", "pre_relief_refinance_loan_seq_no", "relief_refinance_indicator"],
               inplace = True) 

# for metro_code_msa: Null indicates that the area in which the mortgaged property is
# located is a) neither an MSA nor a Metropolitan Division, or b) unknown.

# so we will keep it for EDA purposes and modify the column in order to use it for the project.

### 2. Chcek for duplicated values.

In [9]:
print(orig_2004.duplicated().sum())
#No duplicates

0


In [10]:
# Create a list of 32 generic names: ['col_0', 'col_1', ..., 'col_31']
perf_cols = [f"col_{i}" for i in range(32)]

# Override index 0 (1st column) and index 3 (4th column) with your specific names
perf_cols[0] = "loan_seq_no"
perf_cols[3] = "current_loan_delinquency_status"

## Performance files....

In [12]:
import duckdb

# Define the path to your file
file_path = "D:\\freddie_mac\\data\\2004\\historical_data_time_2004Q1.txt"

# Write standard SQL to read the text file directly
query = f"""
    SELECT 
        column00 AS loan_seq_no, 
        column03 AS current_loan_delinquency_status 
    FROM read_csv_auto('{file_path}', sep='|', header=False)
"""

# Run the query and convert the result directly to a pandas DataFrame
q1p = duckdb.sql(query).df()

In [13]:
q1p.head()

,loan_seq_no,current_loan_delinquency_status
0,F04Q10000001,0
1,F04Q10000001,0
2,F04Q10000001,0
3,F04Q10000001,0
4,F04Q10000001,0


In [14]:
q1p.shape

(40813301, 2)

In [16]:
orig_2004["loan_seq_no"].head()

0    F04Q10000001
1    F04Q10000002
2    F04Q10000003
3    F04Q10000004
4    F04Q10000005
Name: loan_seq_no, dtype: str

In [11]:
# Force pandas to use the 'python' parsing engine to completely bypass the buggy C-parser
q1p = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_time_2004Q1.txt", sep="|", header=None, names=perf_cols, usecols=["loan_seq_no","current_loan_delinquency_status"], engine="python")
q2p = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_time_2004Q2.txt", sep="|", header=None, names=perf_cols, usecols=["loan_seq_no","current_loan_delinquency_status"], engine="python")
q3p = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_time_2004Q3.txt", sep="|", header=None, names=perf_cols, usecols=["loan_seq_no","current_loan_delinquency_status"], engine="python")
q4p = pd.read_csv("D:\\freddie_mac\\data\\2004\\historical_data_time_2004Q4.txt", sep="|", header=None, names=perf_cols, usecols=["loan_seq_no","current_loan_delinquency_status"], engine="python")

KeyboardInterrupt: 

In [ ]:
print(q1p.shape, q2p.shape, q3p.shape, q4p.shape)
print(int(q1p.shape[0])+ int(q2p.shape[0])+ int(q3p.shape[0]) + int(q4p.shape[0]))

In [ ]:
orig_2004.describe()

In [ ]:
orig_2004.dtypes

cred_score                            int64
first_payment_date                    int64
first_time_homebuyer_flag               str
maturity_date                         int64
metro_code_msa                      float64
mortgage_insurance_percent            int64
unit_no                               int64
occupancy_status                        str
og_cltv                               int64
og_dti                                int64
og_upb                                int64
og_ltv                                int64
og_int_rate                         float64
channel                                 str
ppm_flag                                str
amort_type                              str
prop_state                              str
prop_type                               str
postal_code                           int64
loan_seq_no                             str
loan_purpose                            str
og_loan_term                          int64
no_of_borrowers                 

In [ ]:
# Create a DataFrame containing only the categorical columns
cat_df = orig_2004.select_dtypes(include=["object"])

# Loop through its columns to see the unique values for each
for col in cat_df.columns:
    print(f"Unique values for {col}:")
    print(cat_df[col].unique())
    print("-" * 20)